In [1]:
import pandas as pd
from statsmodels.sandbox.tools.cross_val import LeaveOneOut

In [13]:
#df_arduino = pd.read_csv("../ArduinoApps/emg_auswertung/tsfel_debug.csv", index_col=0)
df_full = pd.read_csv("../auswertung_features/cache/feature_trim=0-minlen=0-win=0-step=50-fs=default_tsfel.csv", index_col=0)
df_expect = df_full.iloc[:,6:]

df_lorenzen = pd.read_csv("../auswertung_features/cache/feature_trim=0-minlen=0-win=0-step=50-fs=tsfel.csv", index_col=0)
a = df_lorenzen.filter(regex="^0_*", axis=1)
print(df_lorenzen.iloc[:,6:].shape, a.shape)

(1250, 260) (1250, 65)


In [10]:
# df_a: "normale" Spaltennamen
# df_b: hat ggf. "sensor_" als Präfix

# 1) Spalten von df_b normalisieren (nur wenn sie mit "sensor_" anfangen)
df_b_norm_cols = df_arduino.columns.str.replace(r"^sensor_", "", regex=True)

# Optional: Kopie mit umbenannten Spalten erstellen
df_b_norm = df_arduino.copy()
df_b_norm.columns = df_b_norm_cols

# 2) Vergleich: welche Spalten fehlen wo?
cols_a = set(df_expect.columns)
cols_b = set(df_b_norm.columns)

b = list(df_b_norm.columns)
count = 0
for i, column in  enumerate(df_expect.columns):
       if column != b[i]:
           print(column)
       else:
           count += 1
print(f"Anzahl an gleichen headern: {count}")
fehlen_in_b = sorted(cols_a - cols_b)  # in df_a vorhanden, aber nicht in df_b (nach Normalisierung)
fehlen_in_a = sorted(cols_b - cols_a)  # in df_b vorhanden, aber nicht in df_a

print("Fehlen in df_b:", fehlen_in_b)
print("Anzahl:", len(fehlen_in_b))
print("\n\nFehlen in df_a:", fehlen_in_a)
print("Anzahl:", len(fehlen_in_a))

# 3) (optional) Gemeinsame Spalten
gemeinsam = sorted(cols_a & cols_b)
print("\n\nGemeinsam:", gemeinsam)


KeyboardInterrupt: 

# Test Model on Input again

In [38]:
import pandas as pd
import numpy as np
from auswertung_features.build_features import split_into_label_blocks, load_session
from ArduinoApps.emg_auswertung.python.features import building_feature, block_to_win
from ANN.loso_windowed_blockfeat import compute_W_max

# load dataset
session = load_session("../ArduinoApps/emg_messung/python/messdaten/messung_r_2.csv")
blocks = split_into_label_blocks(session)

test_block = blocks[1][0] # taking the 0 block

feats_df = building_feature(pd.DataFrame(test_block))
win = block_to_win(test_block, T = 500, stride = 250)

W_max = compute_W_max(2000, 500, 250)
X_ts_block, wlen, mask_block = block_to_win(test_block, T = 500, stride = 250, W_max=W_max)

W_max: 7 len(windows): 6 wlen: 6
X_ts_block.shape: (1, 7, 500, 4)
mask_block.shape: (1, 7)


In [39]:
import tensorflow as tf
import pickle

tf_model = tf.keras.models.load_model("models/EMG-CNN-Model.keras")
scaler = pickle.load(open("models/scaler_features.pkl", "rb"))

X_feat = np.asarray(feats_df, dtype=np.float32)
X_feat = scaler.transform(X_feat).astype(np.float32)
X_feat3 = np.repeat(X_feat, repeats=win.shape[0], axis=0)  # (3, 624)

prob = tf_model.predict({"ts": win, "feat": X_feat3}, verbose=0)
y_pred = np.argmax(prob, axis=1)
print(y_pred)

[1 1 1 1 1 1]


In [40]:
from ai_edge_litert.interpreter import Interpreter

lite_scaler =  pickle.load(open("models/scaler_features.pkl", "rb"))
tflite_model = Interpreter(model_path="models/EMG-CNN.tflite")
tflite_model.allocate_tensors()

in_details = tflite_model.get_input_details()
out_details = tflite_model.get_output_details()

N = win.shape[0]
X_feat = np.asarray(feats_df, dtype=np.float32)
X_feat = scaler.transform(X_feat).astype(np.float32)
#X_feat3 = np.repeat(X_feat, repeats=win.shape[0], axis=0)  # (3, 624)

probs = np.zeros((N, 5), dtype=np.float32)

for i in range(N):
    x_ts = win[i:i+1, :, :].astype(np.float32)       # (1,500,4)

    tflite_model.set_tensor(in_details[0]["index"], X_feat)    # (1,624)
    tflite_model.set_tensor(in_details[1]["index"], x_ts)      # (1,500,4)
    tflite_model.invoke()

    probs[i] = tflite_model.get_tensor(out_details[0]["index"])[0]  # (5,)

prob_block = probs.mean(axis=0)
print(int(np.argmax(prob_block)))

1


In [42]:
import tensorflow as tf
import pickle

gru_model = tf.keras.models.load_model("models/EMG-GRU-Model.keras")
scaler_gru = pickle.load(open("models/scaler_features_GRU.pkl", "rb"))

X_feat = np.asarray(feats_df, dtype=np.float32)
X_feat = scaler_gru.transform(X_feat).astype(np.float32)

prob = gru_model.predict({"ts": X_ts_block, "feat": X_feat, "mask":mask_block}, verbose=0)
print(prob)
pred  = int(np.argmax(prob[0]))
print(pred)

[[3.7382273e-03 9.7266668e-01 1.7595258e-04 2.3339057e-02 8.0141799e-05]]
1


# Testing Testing Testing

In [ ]:
import pandas as pd
import numpy as np
from auswertung_features.build_features import split_into_label_blocks, load_session
from sklearn.model_selection import LeaveOneGroupOut

meta = pd.read_csv("../auswertung_features/meta.csv")
dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "rel_path":[], "block_id":[]}
for i, r in meta.iterrows():
    csv_path = r["rel_path"]
    df_session = load_session(csv_path)

    #Error Handling für leere dfs
    if df_session is None:
        continue

    blocks = split_into_label_blocks(df_session)
    for block in blocks:
        meta_blocks["subject_id"].append(r["subject_id"])
        meta_blocks["hand"].append(r["hand"])
        meta_blocks["rel_path"].append(r["rel_path"])
        meta_blocks["block_id"].append(len(dict_blocks))
        dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)
meta_blocks_r = meta_blocks[meta_blocks["hand"]=="r"].reset_index(drop=True)
meta_blocks_l = meta_blocks[meta_blocks["hand"]=="l"].reset_index(drop=True)

[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty



In [3]:
from LOSO_Run_2_input import tsfel_feature
from sklearn.preprocessing import StandardScaler
def logo_baseline(model, meta_block, dict_blocks, feature_set_name= "default_tsfel", b_scaling=True):
    logo = LeaveOneGroupOut()
    groups = meta["subject_id"].to_numpy()
    fold_metrics = []

    for fold, (train_idx, val_idx) in enumerate(logo.split(meta, groups=groups)):
        meta_tr = meta.iloc[train_idx].reset_index(drop=True)
        meta_va = meta.iloc[val_idx].reset_index(drop=True)
        feat_global = tsfel_feature(meta, feature_set_name= feature_set_name)

        feat_cols = feat_global.columns[6:]
        Xf_tr = feat_global.loc[meta_tr["block_id"].astype(int), feat_cols].to_numpy()

        if b_scaling:
            scaler = StandardScaler()
            scaler.fit(Xf_tr)
            X_all = feat_global.loc[:, feat_cols].to_numpy(np.float32)
            X_all_scaled = scaler.transform(X_all)
            X_all.loc[:, feat_cols] = X_all_scaled
        else:
            X_all = feat_global.loc[:,feat_cols].to_numpy(np.float32)

            X_tr = df_tr[feat_cols].to_numpy(np.float32)
    y_tr = df_tr["label"].to_numpy(int)
    X_te = df_te[feat_cols].to_numpy(np.float32)
    y_te = df_te["label"].to_numpy(int)
        model.fit("")

_IncompleteInputError: incomplete input (2701474459.py, line 22)